In [57]:
from datetime import datetime
import shutil
import numpy as np
import cv2
import os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import cm
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

def load_suite2p_data(suite2p_path):
    """Load Suite2p data from the specified path."""
    ops = np.load(os.path.join(suite2p_path, 'ops.npy'), allow_pickle=True).item()
    stat = np.load(os.path.join(suite2p_path, 'stat.npy'), allow_pickle=True)
    F = np.load(os.path.join(suite2p_path, 'F.npy'))
    Fneu = np.load(os.path.join(suite2p_path, 'Fneu.npy'))
    
    # Load registered binary data if available
    reg_file = os.path.join(suite2p_path, 'data.bin')
    if os.path.exists(reg_file):
        reg_data = np.fromfile(reg_file, dtype=np.int16)
        n_frames = ops['nframes']
        Ly, Lx = ops['Ly'], ops['Lx']
        reg_data = reg_data.reshape((n_frames, Ly, Lx))
    else:
        reg_data = None
    
    return ops, stat, F, Fneu, reg_data

def normalize_frame(frame, percentile_min=1, percentile_max=99):
    """Normalize frame to 0-255 range using percentile clipping."""
    vmin = np.percentile(frame, percentile_min)
    vmax = np.percentile(frame, percentile_max)
    frame_norm = np.clip((frame - vmin) / (vmax - vmin), 0, 1)
    return (frame_norm * 255).astype(np.uint8)

def add_scale_bar(frame, ops, pixel_size_um=1.0, scale_bar_um=50):
    """Add a white scale bar with white text to the frame at bottom-right corner."""
    scale_bar_pixels = int(scale_bar_um / pixel_size_um)
    
    # Convert to PIL Image for better text rendering
    pil_img = Image.fromarray(frame)
    draw = ImageDraw.Draw(pil_img)
    
    # Position: bottom-right corner with margins
    margin = 20
    bar_y = frame.shape[0] - margin
    bar_x2 = frame.shape[1] - margin
    bar_x1 = bar_x2 - scale_bar_pixels
    
    # Draw white scale bar
    bar_thickness = 4
    bar_color = (255, 255, 255)
    
    for i in range(bar_thickness):
        draw.line([(bar_x1, bar_y + i - bar_thickness//2), (bar_x2, bar_y + i - bar_thickness//2)], 
                  fill=bar_color, width=1)
    
    # Add white text label with Arial font
    text = f"{scale_bar_um} um"
    try:
        # Try to load Arial font - adjust size as needed
        font = ImageFont.truetype("arial.ttf", 18)
    except:
        # Fallback to default font if Arial not found
        font = ImageFont.load_default()
    
    # Get text bounding box for centering
    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    
    text_x = bar_x1 + (scale_bar_pixels - text_width) // 2
    text_y = bar_y - text_height - 12
    
    draw.text((text_x, text_y), text, fill=bar_color, font=font)
    
    # Convert back to numpy array
    return np.array(pil_img)

def add_timestamp(frame, frame_num, total_frames, fps=30):
    """Add frame counter in white text at the bottom center, outside the video area."""
    # Convert to PIL Image for better text rendering
    pil_img = Image.fromarray(frame)
    draw = ImageDraw.Draw(pil_img)
    
    # Frame counter with Arial font
    text = f"Frame {frame_num}/{total_frames}"
    
    try:
        # Try to load Arial font - larger size for frame counter
        font = ImageFont.truetype("arial.ttf", 36)
    except:
        # Fallback to default font if Arial not found
        font = ImageFont.load_default()
    
    color = (255, 255, 255)
    
    # Get text bounding box for centering
    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    
    # Position: bottom center, below the video
    text_x = (frame.shape[1] - text_width) // 2
    text_y = frame.shape[0] - 45
    
    draw.text((text_x, text_y), text, fill=color, font=font)
    
    # Convert back to numpy array
    return np.array(pil_img)

def create_video_from_suite2p(suite2p_path,
                               fps=30, 
                               pixel_size_um=1.0,
                               scale_bar_um=50,
                               start_frame=0,
                               end_frame=None,
                               codec='mp4v',
                               temporal_bin=1):
    """
    Create a video from Suite2p data with cells flashing in green.
    
    Parameters:
    -----------
    suite2p_path : str
        Path to Suite2p output folder (containing ops.npy, stat.npy, etc.)
    output_path : str
        Output video file path (e.g., 'output.mp4')
    fps : int
        Frames per second for output video
    pixel_size_um : float
        Size of each pixel in micrometers (for scale bar)
    scale_bar_um : float
        Length of scale bar in micrometers
    start_frame : int
        Starting frame index
    end_frame : int or None
        Ending frame index (None = all frames)
    codec : str
        Video codec ('mp4v' for .mp4, 'XVID' for .avi)
    temporal_bin : int
        Number of frames to average together (1 = no averaging, 5 = average every 5 frames)
        Higher values reduce noise but also reduce temporal resolution
    """
    folder_path = suite2p_path
    subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()]
        
    for subfolder in tqdm(subfolders, desc="Processing subfolders"):
        # in the subfolder, find the suite2p folder (usually named 'suite2p/plane0')
        suite2p_path = os.path.join(subfolder, 'plane0')
        # suite2p_path = os.path.join(subfolder, 'suite2p', 'plane0')
        if not os.path.exists(suite2p_path):
            print(f"Suite2p folder not found in {subfolder}, skipping...")
            continue
        
        print("Loading Suite2p data...")
        ops, stat, F, Fneu, reg_data = load_suite2p_data(suite2p_path)
        
        if reg_data is None:
            print("Error: Could not find registered binary data (data.bin)")
            return
        
        n_frames = reg_data.shape[0]
        if end_frame is None:
            end_frame = n_frames
        
        end_frame = min(end_frame, n_frames)
        
        # Calculate number of binned frames
        n_frames_to_process = end_frame - start_frame
        n_output_frames = n_frames_to_process // temporal_bin
        
        print(f"Creating video with {n_output_frames} frames at {fps} fps...")
        print(f"Temporal binning: averaging every {temporal_bin} frames")
        print(f"Image size: {ops['Ly']} x {ops['Lx']} pixels")
    

        basepath = subfolder
        folder_name = os.path.basename(basepath)
        rec_name = folder_name
        
        # create output folder
        date_str = datetime.now().strftime("%Y%m%d")
        output_file_name = f"{date_str}_{rec_name}_CalciumTransients_Frames{start_frame}-{end_frame}.mp4"
        output_path = os.path.join(suite2p_path, output_file_name)

        # try:
        #     if os.path.exists(output_path):
        #         shutil.rmtree(output_path)
        #     os.makedirs(output_path, exist_ok=True)
        # except PermissionError:
        #     print(f"ERROR: Permission denied for folder: {folder_name}")
        #     continue
        # except Exception as e:
        #     print(f"ERROR: Output folder creation failed: {e}")
        #     continue
        
        # Set up video writer
        # Add extra height for the frame counter at the bottom
        text_area_height = 50
        output_height = ops['Ly'] + text_area_height
        output_width = ops['Lx']
        
        fourcc = cv2.VideoWriter_fourcc(*codec)
        out = cv2.VideoWriter(output_path, fourcc, fps, (output_width, output_height), isColor=True)
        
        # Process frames
        for bin_idx in range(n_output_frames):
            if bin_idx % 100 == 0:
                print(f"Processing frame {bin_idx}/{n_output_frames}...")
            
            # Average frames within this bin
            start_idx = start_frame + bin_idx * temporal_bin
            end_idx = start_idx + temporal_bin
            
            # Get frames and average them
            frames_to_avg = reg_data[start_idx:end_idx]
            frame = np.mean(frames_to_avg, axis=0).astype(np.int16)
            
            # Normalize frame
            frame_norm = normalize_frame(frame)
            
            # Convert to RGB (green channel)
            # For "flashing green cells", we'll make bright areas green
            frame_rgb = np.zeros((frame_norm.shape[0], frame_norm.shape[1], 3), dtype=np.uint8)
            frame_rgb[:, :, 1] = frame_norm  # Green channel
            frame_rgb[:, :, 0] = frame_norm // 3  # Slight red for natural look
            frame_rgb[:, :, 2] = frame_norm // 3  # Slight blue for natural look
            
            # Add scale bar with white box
            frame_rgb = add_scale_bar(frame_rgb, ops, pixel_size_um, scale_bar_um)
            
            # Create extended frame with black area at bottom for text
            extended_frame = np.zeros((output_height, output_width, 3), dtype=np.uint8)
            extended_frame[:ops['Ly'], :, :] = frame_rgb
            
            # Add frame counter in the black area at bottom (white text, centered)
            display_frame = start_idx + temporal_bin // 2 - start_frame + 1
            extended_frame = add_timestamp(extended_frame, display_frame, n_frames_to_process, fps)
            
            # Write frame
            out.write(extended_frame)
        
        out.release()
        print(f"Video saved to {output_path}")

# Example usage
if __name__ == "__main__":
    # Modify these paths for your data
    suite2p_path = r"F:\inyoung\B3_Dup_Org3_3x_GZ-001"
    # suite2p_path = r"Z:\from_jasmine\VideoPLZ"
    # suite2p_path = r"\\goard-nas1\Goard_Lab\Jasmine\z_misc\todenoise\New folder (2)"

    create_video_from_suite2p(
        suite2p_path=suite2p_path,
        # fps=10,  # Adjust to match your imaging frame rate
        fps=15.11,  # Adjust to match your imaging frame rate
        pixel_size_um=0.364707528427891,  # Adjust based on your microscope settings
        scale_bar_um=50,  # Length of scale bar
        start_frame=1,
        end_frame=2000,  # Set to None for all frames, or specify a number
        codec='mp4v',  # Use 'XVID' for .avi files
        temporal_bin=3 # Average every x frames to reduce noise
    )

Processing subfolders:   0%|          | 0/2 [00:00<?, ?it/s]

Suite2p folder not found in F:\inyoung\B3_Dup_Org3_3x_GZ-001\References, skipping...
Loading Suite2p data...
Creating video with 333 frames at 15.11 fps...
Temporal binning: averaging every 3 frames
Image size: 1024 x 1024 pixels
Processing frame 0/333...
Processing frame 100/333...
Processing frame 200/333...
Processing frame 300/333...


Processing subfolders: 100%|██████████| 2/2 [00:14<00:00,  7.45s/it]

Video saved to F:\inyoung\B3_Dup_Org3_3x_GZ-001\suite2p\plane0\20251120_suite2p_CalciumTransients_Frames1-1000.mp4
